In [ ]:
# ============================================================
# INSTALL LIBRARY
# ============================================================

!pip install transformers torch -q

# ============================================================
# IMPORT
# ============================================================

import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv('/content/dataset_labeled_berita_withscore.csv')

# ============================================================
# GABUNGKAN TITLE + CONTENT
# ============================================================

df['text'] = (
    df['title'].fillna('') + ' ' +
    df['content'].fillna('')
)

# ============================================================
# LOAD MODEL INDOBERT / INDOROBERTA
# ============================================================

model_name = "w11wo/indonesian-roberta-base-sentiment-classifier"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# ============================================================
# LABEL MAPPING
# ============================================================

id2label = {
    0: 'negative',
    1: 'neutral',
    2: 'positive'
}

# ============================================================
# PREDICTION
# ============================================================

predictions = []

model.eval()

for text in tqdm(df['text']):

    inputs = tokenizer(
        str(text),
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors='pt'
    )

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()

    predictions.append(id2label[pred])

# ============================================================
# SAVE HASIL
# ============================================================

df['sentiment_indobert'] = predictions

df.to_csv(
    'dataset_berita_indobert_relabel.csv',
    index=False
)

# ============================================================
# CEK DISTRIBUSI
# ============================================================

print(df['sentiment_indobert'].value_counts())

print("\nCSV berhasil disimpan!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: w11wo/indonesian-roberta-base-sentiment-classifier
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 2061/2061 [34:36<00:00,  1.01s/it]


sentiment_indobert
neutral     1224
positive     768
negative      69
Name: count, dtype: int64

CSV berhasil disimpan!
